### Links and Resources
| Concept | Reference |
|---|---|
| [`CREATE FUNCTION` (SQL)](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-sql-function) | DDL syntax |
| [Unity Catalog functions](https://docs.databricks.com/aws/en/udf/unity-catalog) | Governance model |
| [`GRANT` on functions](https://docs.databricks.com/aws/en/sql/language-manual/security-grant) | Permissions |
| [Built-in function index](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-functions-builtin) | What's already there |

In [ ]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel \u2014 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

from pyspark.sql import functions as F


def display(df, n=20):
    df.show(n, truncate=False)

# SQL UDFs in Unity Catalog

A SQL UDF is a function whose body is a **single SQL expression**. Unlike Python UDFs, it:

- Lives in Unity Catalog at `catalog.schema.function_name`.
- Is governed by `GRANT` / `REVOKE` like a table.
- Can be **inlined by Catalyst** — at execution time it disappears into the surrounding query plan, so there's **no performance penalty** vs writing the expression by hand.
- Is callable from any notebook, SQL warehouse, or BI tool that has permission.

It's the right tool for *reusable business logic* expressed in SQL: tax calculations, status mappings, normalization rules. **All cells below run as SQL** — change the catalog/schema names to match your workspace.

## Set the working catalog/schema

Replace these with names you can write to. The defaults below match Databricks' built-in `main.default`.

In [ ]:
CATALOG = "main"
SCHEMA  = "default"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA  {SCHEMA}")
print("using:", spark.sql("SELECT current_catalog(), current_schema()").collect()[0])

## Create a scalar SQL UDF

`CREATE FUNCTION ... RETURNS T RETURN <single SQL expression>`. The expression can reference parameters and any built-in function — but **not another DataFrame query**.

In [ ]:
spark.sql("""
CREATE OR REPLACE FUNCTION clean_email(e STRING)
    RETURNS STRING
    COMMENT 'Lowercase, trim, mask domain after @'
    RETURN
        CASE
            WHEN e IS NULL THEN NULL
            WHEN e RLIKE '^[^@]+@[^@]+$' THEN lower(trim(regexp_replace(e, '@.*$', '@REDACTED')))
            ELSE NULL  -- malformed
        END
""")
print("created clean_email")

## Use the UDF anywhere SQL runs

In [ ]:
spark.sql("""
SELECT raw,
       clean_email(raw) AS cleaned
FROM (VALUES ('  Alice@Example.COM '), ('not-an-email'), (NULL))
   AS t(raw)
""").show(truncate=False)

## Multi-parameter UDF — bucketing logic

In [ ]:
spark.sql("""
CREATE OR REPLACE FUNCTION risk_band(score INT, country STRING)
    RETURNS STRING
    RETURN
        CASE
            WHEN score IS NULL                          THEN 'unknown'
            WHEN country IN ('US', 'CA') AND score < 600 THEN 'high'
            WHEN country IN ('US', 'CA') AND score < 750 THEN 'medium'
            WHEN country IN ('US', 'CA')                 THEN 'low'
            WHEN score < 500                             THEN 'high'
            WHEN score < 650                             THEN 'medium'
            ELSE 'low'
        END
""")

spark.sql("""
SELECT * FROM (VALUES
    (820, 'US'), (640, 'US'), (550, 'CA'),
    (700, 'BE'), (480, 'FR'), (NULL, 'DE')
) AS t(score, country)
ORDER BY score
""").withColumn("band", F.expr("risk_band(score, country)")).show()

## Table-valued SQL UDFs (TVFs) — `RETURNS TABLE`

A TVF returns rows, not a scalar. You call it in the `FROM` clause, optionally with `LATERAL` to correlate with each outer row.

In [ ]:
spark.sql("""
CREATE OR REPLACE FUNCTION date_buckets(start_d DATE, n INT)
    RETURNS TABLE (d DATE, bucket STRING)
    RETURN
        SELECT
            date_add(start_d, x) AS d,
            CASE WHEN dayofweek(date_add(start_d, x)) IN (1, 7) THEN 'weekend' ELSE 'weekday' END AS bucket
        FROM (SELECT explode(sequence(0, n - 1)) AS x)
""")

spark.sql("SELECT * FROM date_buckets(DATE '2024-01-01', 10)").show()

## Inspecting and managing functions

In [ ]:
spark.sql("SHOW USER FUNCTIONS").show(20, truncate=False)
spark.sql("DESCRIBE FUNCTION EXTENDED clean_email").show(20, truncate=False)

## Permissions — grant before others can use it

In Unity Catalog, even a function you created is invisible to other users until you grant on it (and on the parent schema/catalog).

In [ ]:
# Replace the principal with a real user/group/service principal in your workspace.
# Commented because the principal name is workspace-specific.

# spark.sql("GRANT EXECUTE ON FUNCTION clean_email TO `analysts@example.com`")
# spark.sql("GRANT USAGE   ON SCHEMA   default       TO `analysts@example.com`")
# spark.sql("GRANT USAGE   ON CATALOG  main          TO `analysts@example.com`")
print("see comments — fill in your principal")

## Why SQL UDFs are essentially free

Unity Catalog SQL UDFs are **inlined** during query optimization. The plan for:

```sql
SELECT clean_email(raw) FROM customers
```

is identical to writing the `CASE`/`regexp_replace` chain inline. You get the *encapsulation* benefit (one definition, audit-able, granted) without paying the UDF tax.

You can confirm with `EXPLAIN`:

In [ ]:
spark.sql("EXPLAIN SELECT clean_email(raw) FROM (VALUES ('a@b.com')) t(raw)").show(truncate=False)

## Limitations

- **Single SQL expression** — no statements, no loops, no `DECLARE`. For multi-step logic, build it from sub-expressions or compose multiple UDFs.
- **No DML inside the body** — purely a function over its inputs.
- **TVFs cannot reference outer correlation directly** unless invoked with `LATERAL`.
- **Full namespace** — to call from another schema, use `catalog.schema.fn(...)` or `USE CATALOG/SCHEMA` first.

## Cleanup

In [ ]:
spark.sql("DROP FUNCTION IF EXISTS clean_email")
spark.sql("DROP FUNCTION IF EXISTS risk_band")
spark.sql("DROP FUNCTION IF EXISTS date_buckets")
print("dropped")

## Takeaways

- SQL UDFs are governed objects in Unity Catalog — name, comment, permission them like tables.
- Single-expression body, but that expression can be arbitrarily complex (CASE, regex, struct literals, all built-ins).
- Inlined by Catalyst → **zero runtime overhead** vs writing the SQL inline.
- Use `RETURNS TABLE` for TVFs; call them in `FROM`, optionally with `LATERAL`.
- Grant `EXECUTE` plus `USAGE` on the parent schema/catalog before sharing.